In [1]:
# Import required packages
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

print(f"Using {os.cpu_count()} CPU cores for parallel cross-validation")

c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Using 32 CPU cores for parallel cross-validation


In [2]:
# Directories
PROJECT_DIR = r"C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/"

CODE_DIR = os.path.join(PROJECT_DIR, "Code")
DATA_DIR = os.path.join(PROJECT_DIR, "Data")
FIGURES_DIR = os.path.join(PROJECT_DIR, "Figures")
TABLES_DIR = os.path.join(PROJECT_DIR, "Tables")

# File names
INPUT_DATA = os.path.join(DATA_DIR, "merged_master.pkl")

In [3]:
data = pd.read_pickle(INPUT_DATA)

In [4]:
# Prepare features and target
# Target variable
TARGET = 'f_cumret1'

# Select features
FEATURES = ['net_sentiment', 'log_volume']

# Remove missing values
model_data = data[[TARGET] + FEATURES].dropna()

# Elastic Net parameters
# l1_ratio = 0.5 means equal mix of L1 (LASSO) and L2 (Ridge) penalties
# l1_ratio = 1 is pure LASSO, l1_ratio = 0 is pure Ridge
L1_RATIO = 0.5

print(f"Sample size: {len(model_data):,}")
print(f"Target: {TARGET}")
print(f"Features: {FEATURES}")
print(f"L1 ratio (LASSO vs Ridge mix): {L1_RATIO}")

Sample size: 16,743,676
Target: f_cumret1
Features: ['net_sentiment', 'log_volume']
L1 ratio (LASSO vs Ridge mix): 0.5


# In-Sample Elastic Net regression

In [5]:
# Prepare X and y
X = model_data[FEATURES]
y = model_data[TARGET]

# Normalize features using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit Elastic Net regression model with cross-validation for optimal alpha
# l1_ratio is fixed at 0.5 (equal mix of L1 and L2 penalties)
# max_iter increased to 10000 to ensure convergence
enet_model = ElasticNetCV(l1_ratio=L1_RATIO, cv=5, random_state=42, n_jobs=-1, max_iter=10000)
enet_model.fit(X_scaled, y)

# Make predictions
y_pred = enet_model.predict(X_scaled)

# Evaluate performance
r2 = r2_score(y, y_pred)
mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)

print("In-Sample Elastic Net Regression Results")
print("=" * 50)
print(f"L1 ratio (fixed): {L1_RATIO}")
print(f"Optimal alpha (regularization): {enet_model.alpha_:.6f}")
print(f"R-squared: {r2:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MSE: {mse:.6f}")
print("\nCoefficients (on standardized features):")
for feature, coef in zip(FEATURES, enet_model.coef_):
    print(f"  {feature}: {coef:.6f}")
print(f"  Intercept: {enet_model.intercept_:.6f}")

In-Sample Elastic Net Regression Results
L1 ratio (fixed): 0.5
Optimal alpha (regularization): 0.000070
R-squared: 0.000078
RMSE: 0.047094
MSE: 0.002218

Coefficients (on standardized features):
  net_sentiment: 0.000206
  log_volume: -0.000431
  Intercept: 0.000496


# OOS predictions

In [6]:
# Out-of-sample predictions with MONTHLY TRAINING but DAILY PREDICTIONS
# The model is trained once per month (at month-end) and used to predict all days in the following month
# Parameters
TRAIN_END_DATE = '2011-12-31'
WINDOW_252 = 252  # One trading year (in days) for rolling window
WINDOW_21 = 21    # One trading month (in days) for rolling window
MAX_ITER = 10000  # Increased iterations for convergence

# Ensure date column is datetime
model_data['date'] = pd.to_datetime(data.loc[model_data.index, 'date'])

# Sort by date
model_data = model_data.sort_values('date')

# Get unique dates
unique_dates = model_data['date'].unique()
unique_dates = pd.DatetimeIndex(unique_dates).sort_values()

# Find the starting prediction date (first date after training window)
train_end = pd.to_datetime(TRAIN_END_DATE)
oos_dates = unique_dates[unique_dates > train_end]

# Create a year-month column for grouping
model_data['year_month'] = model_data['date'].dt.to_period('M')

# Get unique months in the OOS period
oos_months = model_data.loc[model_data['date'] > train_end, 'year_month'].unique()
oos_months = sorted(oos_months)

print(f"Training window: {unique_dates[0].strftime('%Y-%m-%d')} to {TRAIN_END_DATE}")
print(f"OOS prediction period: {oos_dates[0].strftime('%Y-%m-%d')} to {oos_dates[-1].strftime('%Y-%m-%d')}")
print(f"Number of OOS dates: {len(oos_dates):,}")
print(f"Number of OOS months: {len(oos_months)}")

# Initialize storage for predictions and selected alphas
predictions_expanding = []
predictions_rolling_252 = []
predictions_rolling_21 = []
alphas_expanding = []
alphas_rolling_252 = []
alphas_rolling_21 = []

# Loop through OOS months (train once per month)
for month_idx, pred_month in enumerate(oos_months):
    # Get all prediction dates in this month
    month_dates = oos_dates[oos_dates.to_period('M') == pred_month]

    if len(month_dates) == 0:
        continue

    # Training cutoff: end of the previous month (first day of pred_month - 1 day)
    first_day_of_month = pred_month.to_timestamp()
    train_cutoff = first_day_of_month - pd.Timedelta(days=2)

    # Find the last trading day before or on train_cutoff
    train_dates = unique_dates[unique_dates <= train_cutoff]
    if len(train_dates) == 0:
        continue
    last_train_date = train_dates[-1]

    # 1. EXPANDING WINDOW: Train on all data up to end of previous month
    train_mask_exp = model_data['date'] <= last_train_date
    X_train_exp = model_data.loc[train_mask_exp, FEATURES]
    y_train_exp = model_data.loc[train_mask_exp, TARGET]

    if len(X_train_exp) > 0:
        # Normalize features
        scaler_exp = StandardScaler()
        X_train_exp_scaled = scaler_exp.fit_transform(X_train_exp)
        
        # Fit Elastic Net with CV (fixed l1_ratio)
        enet_exp = ElasticNetCV(l1_ratio=L1_RATIO, cv=5, random_state=42, n_jobs=-1, max_iter=MAX_ITER)
        enet_exp.fit(X_train_exp_scaled, y_train_exp)
        alphas_expanding.append({'month': pred_month, 'alpha': enet_exp.alpha_})

        # Use this model to predict for all days in the month
        for pred_date in month_dates:
            test_mask = model_data['date'] == pred_date
            X_test = model_data.loc[test_mask, FEATURES]

            if len(X_test) == 0:
                continue

            # Transform test features using the same scaler
            X_test_scaled = scaler_exp.transform(X_test)
            
            test_indices = model_data.index[test_mask]
            y_pred_exp = enet_exp.predict(X_test_scaled)

            for idx, pred in zip(test_indices, y_pred_exp):
                predictions_expanding.append({
                    'date': pred_date,
                    'index': idx,
                    'pred_expanding': pred
                })

    # 2. ROLLING 252-DAY WINDOW: Train on last 252 trading days before the month
    last_train_date_idx = unique_dates.get_loc(last_train_date)
    if last_train_date_idx >= WINDOW_252:
        start_date_252 = unique_dates[last_train_date_idx - WINDOW_252 + 1]
        train_mask_252 = (model_data['date'] >= start_date_252) & (model_data['date'] <= last_train_date)
        X_train_252 = model_data.loc[train_mask_252, FEATURES]
        y_train_252 = model_data.loc[train_mask_252, TARGET]

        if len(X_train_252) > 0:
            # Normalize features
            scaler_252 = StandardScaler()
            X_train_252_scaled = scaler_252.fit_transform(X_train_252)
            
            # Fit Elastic Net with CV (fixed l1_ratio)
            enet_252 = ElasticNetCV(l1_ratio=L1_RATIO, cv=5, random_state=42, n_jobs=-1, max_iter=MAX_ITER)
            enet_252.fit(X_train_252_scaled, y_train_252)
            alphas_rolling_252.append({'month': pred_month, 'alpha': enet_252.alpha_})

            # Use this model to predict for all days in the month
            for pred_date in month_dates:
                test_mask = model_data['date'] == pred_date
                X_test = model_data.loc[test_mask, FEATURES]

                if len(X_test) == 0:
                    continue

                # Transform test features using the same scaler
                X_test_scaled = scaler_252.transform(X_test)
                
                test_indices = model_data.index[test_mask]
                y_pred_252 = enet_252.predict(X_test_scaled)

                for idx, pred in zip(test_indices, y_pred_252):
                    predictions_rolling_252.append({
                        'date': pred_date,
                        'index': idx,
                        'pred_rolling_252': pred
                    })

    # 3. ROLLING 21-DAY WINDOW: Train on last 21 trading days before the month
    if last_train_date_idx >= WINDOW_21:
        start_date_21 = unique_dates[last_train_date_idx - WINDOW_21 + 1]
        train_mask_21 = (model_data['date'] >= start_date_21) & (model_data['date'] <= last_train_date)
        X_train_21 = model_data.loc[train_mask_21, FEATURES]
        y_train_21 = model_data.loc[train_mask_21, TARGET]

        if len(X_train_21) > 0:
            # Normalize features
            scaler_21 = StandardScaler()
            X_train_21_scaled = scaler_21.fit_transform(X_train_21)
            
            # Fit Elastic Net with CV (fixed l1_ratio)
            enet_21 = ElasticNetCV(l1_ratio=L1_RATIO, cv=5, random_state=42, n_jobs=-1, max_iter=MAX_ITER)
            enet_21.fit(X_train_21_scaled, y_train_21)
            alphas_rolling_21.append({'month': pred_month, 'alpha': enet_21.alpha_})

            # Use this model to predict for all days in the month
            for pred_date in month_dates:
                test_mask = model_data['date'] == pred_date
                X_test = model_data.loc[test_mask, FEATURES]

                if len(X_test) == 0:
                    continue

                # Transform test features using the same scaler
                X_test_scaled = scaler_21.transform(X_test)
                
                test_indices = model_data.index[test_mask]
                y_pred_21 = enet_21.predict(X_test_scaled)

                for idx, pred in zip(test_indices, y_pred_21):
                    predictions_rolling_21.append({
                        'date': pred_date,
                        'index': idx,
                        'pred_rolling_21': pred
                    })

    # Progress update
    if (month_idx + 1) % 12 == 0:
        print(f"Processed {month_idx + 1}/{len(oos_months)} months ({100 * (month_idx + 1) / len(oos_months):.1f}%)")

print(f"\nCompleted.")
print(f"Expanding window predictions: {len(predictions_expanding):,}")
print(f"Rolling 252-day predictions: {len(predictions_rolling_252):,}")
print(f"Rolling 21-day predictions: {len(predictions_rolling_21):,}")

Training window: 2010-01-04 to 2011-12-31
OOS prediction period: 2012-01-03 to 2024-12-30
Number of OOS dates: 3,269
Number of OOS months: 156
Processed 12/156 months (7.7%)
Processed 24/156 months (15.4%)
Processed 36/156 months (23.1%)
Processed 48/156 months (30.8%)
Processed 60/156 months (38.5%)
Processed 72/156 months (46.2%)
Processed 84/156 months (53.8%)
Processed 96/156 months (61.5%)
Processed 108/156 months (69.2%)
Processed 120/156 months (76.9%)
Processed 132/156 months (84.6%)
Processed 144/156 months (92.3%)
Processed 156/156 months (100.0%)

Completed.
Expanding window predictions: 14,545,760
Rolling 252-day predictions: 14,545,760
Rolling 21-day predictions: 14,545,760


In [7]:
# Summary of selected alpha values
print("Selected Alpha (Regularization Parameter) Summary")
print(f"L1 ratio (fixed): {L1_RATIO}")
print("=" * 50)

if alphas_expanding:
    alphas_exp_df = pd.DataFrame(alphas_expanding)
    print(f"\nExpanding Window:")
    print(f"  Mean alpha: {alphas_exp_df['alpha'].mean():.6f}")
    print(f"  Std alpha: {alphas_exp_df['alpha'].std():.6f}")
    print(f"  Min alpha: {alphas_exp_df['alpha'].min():.6f}")
    print(f"  Max alpha: {alphas_exp_df['alpha'].max():.6f}")

if alphas_rolling_252:
    alphas_252_df = pd.DataFrame(alphas_rolling_252)
    print(f"\nRolling 252-day Window:")
    print(f"  Mean alpha: {alphas_252_df['alpha'].mean():.6f}")
    print(f"  Std alpha: {alphas_252_df['alpha'].std():.6f}")
    print(f"  Min alpha: {alphas_252_df['alpha'].min():.6f}")
    print(f"  Max alpha: {alphas_252_df['alpha'].max():.6f}")

if alphas_rolling_21:
    alphas_21_df = pd.DataFrame(alphas_rolling_21)
    print(f"\nRolling 21-day Window:")
    print(f"  Mean alpha: {alphas_21_df['alpha'].mean():.6f}")
    print(f"  Std alpha: {alphas_21_df['alpha'].std():.6f}")
    print(f"  Min alpha: {alphas_21_df['alpha'].min():.6f}")
    print(f"  Max alpha: {alphas_21_df['alpha'].max():.6f}")

Selected Alpha (Regularization Parameter) Summary
L1 ratio (fixed): 0.5

Expanding Window:
  Mean alpha: 0.000122
  Std alpha: 0.000209
  Min alpha: 0.000000
  Max alpha: 0.000725

Rolling 252-day Window:
  Mean alpha: 0.000122
  Std alpha: 0.000252
  Min alpha: 0.000000
  Max alpha: 0.001936

Rolling 21-day Window:
  Mean alpha: 0.000556
  Std alpha: 0.000694
  Min alpha: 0.000000
  Max alpha: 0.003290


In [8]:
# Convert predictions to DataFrames and merge
df_expanding = pd.DataFrame(predictions_expanding)
df_rolling_252 = pd.DataFrame(predictions_rolling_252)
df_rolling_21 = pd.DataFrame(predictions_rolling_21)

# Merge all predictions together
predictions_df = df_expanding.merge(
    df_rolling_252,
    on=['date', 'index'],
    how='outer'
).merge(
    df_rolling_21,
    on=['date', 'index'],
    how='outer'
)

# Add symbol information
predictions_df = predictions_df.merge(
    data[['permno', 'ticker']].reset_index(),
    left_on='index',
    right_on='index',
    how='left'
)

# Reorder columns
predictions_df = predictions_df[['date', 'permno', 'ticker', 'index', 'pred_expanding', 'pred_rolling_252', 'pred_rolling_21']]

# Sort by date and ticker
predictions_df = predictions_df.sort_values(['date', 'ticker']).reset_index(drop=True)

print("Predictions Summary")
print("=" * 50)
print(f"Total observations: {len(predictions_df):,}")
print(f"\nNon-null predictions by window type:")
print(f"  Expanding: {predictions_df['pred_expanding'].notna().sum():,}")
print(f"  Rolling 252-day: {predictions_df['pred_rolling_252'].notna().sum():,}")
print(f"  Rolling 21-day: {predictions_df['pred_rolling_21'].notna().sum():,}")
print(f"\nFirst 10 predictions:")
print(predictions_df.head(10))

Predictions Summary
Total observations: 14,545,760

Non-null predictions by window type:
  Expanding: 14,545,760
  Rolling 252-day: 14,545,760
  Rolling 21-day: 14,545,760

First 10 predictions:
        date  permno ticker    index  pred_expanding  pred_rolling_252  \
0 2012-01-03   87432      A  2879304        0.000502         -0.000204   
1 2012-01-03   24643     AA  2377768        0.000502         -0.000204   
2 2012-01-03   12479    AAC  2266267        0.000502         -0.000204   
3 2012-01-03   90020   AACC  2994307        0.000502         -0.000204   
4 2012-01-03   15580   AAME  2343833        0.000502         -0.000204   
5 2012-01-03   10517    AAN  2211052        0.000502         -0.000204   
6 2012-01-03   76868   AAON  2576068        0.000502         -0.000204   
7 2012-01-03   89217    AAP  2947508        0.000502         -0.000204   
8 2012-01-03   14593   AAPL  2339391       -0.000617         -0.000204   
9 2012-01-03   90854   AATI  3056541        0.000502         -0.0

In [9]:
# Save predictions to Data directory
OUTPUT_DATA_DIR = os.path.join(PROJECT_DIR, "Data")
os.makedirs(OUTPUT_DATA_DIR, exist_ok=True)

OUTPUT_FILE = os.path.join(OUTPUT_DATA_DIR, "predictions_elasticnet.pkl")
predictions_df.to_pickle(OUTPUT_FILE)

print(f"Predictions saved to: {OUTPUT_FILE}")
print(f"File size: {os.path.getsize(OUTPUT_FILE) / (1024**2):.2f} MB")
print(f"Shape: {predictions_df.shape}")

Predictions saved to: C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/Data\predictions_elasticnet.pkl
File size: 757.55 MB
Shape: (14545760, 7)
